# Lab 10 – Adaptive Median Filter and Otsu Thresholding

Add salt-and-pepper noise to the grayscale image. Compare standard and adaptive median filters using PSNR. Then segment the image using Otsu's global threshold.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

image_path = '/content/pexels-chevanon-1108099.jpg'
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

if image is None:
    raise FileNotFoundError('Image not found. Check the image path.')

plt.imshow(image, cmap='gray')
plt.title('Original Grayscale Image')
plt.axis('off')
plt.show()

## 1. Add Salt-and-Pepper Noise

In [ ]:
probability = 0.2
random_values = np.random.rand(*image.shape)
noisy_image = image.copy()
noisy_image[random_values < probability / 2] = 0
noisy_image[(random_values >= probability / 2) & (random_values < probability)] = 255

plt.imshow(noisy_image, cmap='gray')
plt.title('Image with Salt-and-Pepper Noise')
plt.axis('off')
plt.show()

## 2. Standard Median and Adaptive Median Filtering

Enter the initial and maximum window sizes. Both values should be odd.

In [ ]:
initial_size = int(input('Enter initial window size (e.g. 3): '))
maximum_size = int(input('Enter maximum window size (e.g. 7): '))

if initial_size < 3 or initial_size % 2 == 0 or maximum_size < initial_size or maximum_size % 2 == 0:
    raise ValueError('Enter valid odd window sizes, with maximum size >= initial size.')

standard_median = cv2.medianBlur(noisy_image, initial_size)

height, width = noisy_image.shape
pad = maximum_size // 2
padded_image = cv2.copyMakeBorder(noisy_image, pad, pad, pad, pad, cv2.BORDER_REPLICATE)
adaptive_median = np.zeros_like(noisy_image)

for i in range(height):
    for j in range(width):
        current_pixel = int(padded_image[i + pad, j + pad])
        window_size = initial_size
        assigned = False

        while window_size <= maximum_size:
            half = window_size // 2
            window = padded_image[i + pad - half:i + pad + half + 1, j + pad - half:j + pad + half + 1]

            z_min = int(np.min(window))
            z_max = int(np.max(window))
            z_median = int(np.median(window))

            if z_min < z_median < z_max:
                if z_min < current_pixel < z_max:
                    adaptive_median[i, j] = current_pixel
                else:
                    adaptive_median[i, j] = z_median
                assigned = True
                break

            window_size = window_size + 2

        if assigned == False:
            adaptive_median[i, j] = z_median

## 3. Compare the Filters using PSNR

In [ ]:
def calculate_psnr(original, filtered):
    mse = np.mean((original.astype(np.float64) - filtered.astype(np.float64)) ** 2)
    if mse == 0:
        return 100
    return 10 * np.log10((255 * 255) / mse)

psnr_standard = calculate_psnr(image, standard_median)
psnr_adaptive = calculate_psnr(image, adaptive_median)

print('PSNR of Standard Median Filter:', round(psnr_standard, 2), 'dB')
print('PSNR of Adaptive Median Filter:', round(psnr_adaptive, 2), 'dB')

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(noisy_image, cmap='gray')
plt.title('Noisy Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(standard_median, cmap='gray')
plt.title('Standard Median')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(adaptive_median, cmap='gray')
plt.title('Adaptive Median')
plt.axis('off')
plt.show()

## 4. Otsu's Global Thresholding

In [ ]:
histogram, bins = np.histogram(image.flatten(), 256, [0, 256])
probability = histogram / image.size
intensity_values = np.arange(256)
mean_total = np.sum(intensity_values * probability)

maximum_variance = 0
best_threshold = 0
background_weight = 0
background_sum = 0

for t in range(256):
    background_weight = background_weight + probability[t]
    background_sum = background_sum + t * probability[t]
    foreground_weight = 1 - background_weight

    if background_weight == 0 or foreground_weight == 0:
        continue

    mean_background = background_sum / background_weight
    mean_foreground = (mean_total - background_sum) / foreground_weight
    variance = background_weight * foreground_weight * (mean_background - mean_foreground) ** 2

    if variance > maximum_variance:
        maximum_variance = variance
        best_threshold = t

segmented_image = np.where(image >= best_threshold, 255, 0).astype(np.uint8)
print('Otsu Threshold:', best_threshold)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(image, cmap='gray')
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.hist(image.flatten(), 256, [0, 256])
plt.axvline(best_threshold, color='red', linestyle='dashed')
plt.title('Image Histogram')

plt.subplot(1, 3, 3)
plt.imshow(segmented_image, cmap='gray')
plt.title('Otsu Segmented Image')
plt.axis('off')
plt.show()